# CR1 — generation on the GPU (nothing is scored here)

Registered in `research/cr1/PREREG_CR1.md`. For the absolute-realism profile of
controlled output: **14 control states × 5 generations × 3,000 compliant URLs**
from E7, enforced by resampling (`uie.presence.generate_with_presence`, mode
**joint**). The two phishing states with a query and a fixed path choice are
registered as not measurable and are not generated. Scoring happens on the
laptop, with gate v2's own functions.

**Attach three inputs** (right panel → *Add Input*):

| input | what | found by |
|---|---|---|
| `uie-corpus` | the data bundle (only its suffix table is used) | `bundle.manifest.json` |
| **`uie-cr1-code`** — NEW | the current `uie`, with `presence.py` | `CR1_CODE_MANIFEST.json` |
| your E7 training notebook's output | `best.pt` | its SHA-256 |

**Settings:** GPU T4 x2, Internet **On**. **Save & Run All.** About 1.5 h.

In [ ]:
# --- 1. locate the three inputs ------------------------------------------------
import hashlib
from pathlib import Path

E7_SHA256 = "916d03857d7d3db4101d1b106293a83905f4b317676ee7e144b018170cb36065"


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


root = Path("/kaggle/input")
data_hits = sorted(root.rglob("bundle.manifest.json")) if root.exists() else []
code_hits = sorted(root.rglob("CR1_CODE_MANIFEST.json")) if root.exists() else []
pts = {p: sha256(p) for p in sorted(root.rglob("*.pt"))} if root.exists() else {}
ckpt_hits = [p for p, h in pts.items() if h == E7_SHA256]
for p, h in pts.items():
    print(f"   checkpoint seen: {p}  sha256 {h[:16]}")
if not data_hits or not code_hits or not ckpt_hits:
    raise SystemExit(
        "Attach all three inputs (right panel -> Add Input):\n"
        "  uie-corpus    -- has bundle.manifest.json\n"
        "  uie-cr1-code  -- has CR1_CODE_MANIFEST.json (the NEW code dataset)\n"
        "  the E7 training notebook's OUTPUT -- best.pt, sha256 "
        f"{E7_SHA256[:16]}...\nInputs mount when the session starts; rerun after attaching.")
DATA, CODE, CKPT = data_hits[0].parent, code_hits[0].parent, ckpt_hits[0]
print("data:", DATA)
print("code:", CODE)
print("ckpt:", CKPT, "(sha256 verified: best_e7)")

In [ ]:
# --- 2. environment: install first, then run THIS code and prove it ------------
import importlib, json, subprocess, sys
from importlib import metadata

TLDEXTRACT = "5.3.1"


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: Settings -> Internet -> On.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")
print("tldextract:", TLDEXTRACT)

sys.path.insert(0, str(CODE))
for name in [m for m in sys.modules if m == "uie" or m.startswith("uie.")]:
    del sys.modules[name]
import uie
manifest = json.loads((CODE / "CR1_CODE_MANIFEST.json").read_text())
print("uie imported from:", uie.__file__)
assert str(CODE) in uie.__file__, "the CR1 code is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items()
       if hashlib.sha256((CODE / f).read_bytes()).hexdigest() != h]
print(f"code integrity: {len(manifest['files']) - len(bad)}/{len(manifest['files'])} files match")
assert not bad, f"files differ from the manifest: {bad}"
from uie.presence import PresenceUnmet, generate_with_presence
print(f"code: commit {manifest['commit'][:12]}; uie.presence present")

import torch
print("torch:", torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then rerun.")
print("gpu  :", torch.cuda.get_device_name(0))

In [ ]:
# --- 3. the registered states and seeds; inputs they must match ---------------
from uie.suffixes import SuffixTable
from uie.train import load_checkpoint

MODE = "joint"                       # PREREG_CR1: HC2's adopted default
G, N_MAIN = 5, 3000
DELTA = {"benign": 0.22, "phishing": 0.30}
TABLE_SHA256 = "aaa6fb853ea538d41ba4c10661b89faad2a687b869208772c096c7e507ee866d"
OUT = Path("/kaggle/working/cr1_gen")

STATES = ([(label, {f: want}) for label in ("benign", "phishing")
           for f in ("PTH", "QRY") for want in (True, False)]
          + [(label, {"PTH": p, "QRY": q}) for label in ("benign", "phishing")
             for p in (True, False) for q in (True, False)])
NOT_MEASURABLE = {("phishing", (("PTH", True), ("QRY", True))),
                  ("phishing", (("PTH", False), ("QRY", True)))}


def tag_of(label, state):
    return label + "_" + "_".join(f"{f[0]}{int(v)}" for f, v in state.items())


def gen_seed(s, g):
    return 20261001 + (100 * s + g) * 1_000_000_000


assert sha256(DATA / "suffix_table.json") == TABLE_SHA256, "suffix table differs -- STOP"
table = SuffixTable.load(DATA / "suffix_table.json")
device = torch.device("cuda")
model, tokenizer, _ = load_checkpoint(CKPT, device=device)
model.to(device).eval()
where = next(model.parameters()).device
assert where.type == "cuda", f"model is on {where}"
assert getattr(tokenizer, "include_fields", False) and tokenizer.vocab_size == 107
jobs = [(s, label, state, g) for s, (label, state) in enumerate(STATES)
        if (label, tuple(sorted(state.items()))) not in NOT_MEASURABLE for g in range(G)]
print(f"model parameters on {where}; mode {MODE}; {len(jobs)} files to make "
      f"({len(jobs) // G} states x {G} generations x {N_MAIN} compliant URLs)")

In [ ]:
# --- 4. generate: resampling until every URL honours its state -----------------
import time

OUT.mkdir(parents=True, exist_ok=True)
started, records = time.time(), []
for k, (s, label, state, g) in enumerate(jobs, 1):
    path = OUT / f"{tag_of(label, state)}_g{g}.json"
    t0, stats, error = time.time(), {}, None
    if path.is_file():
        urls = json.loads(path.read_text(encoding="utf-8"))
    else:
        try:
            urls = generate_with_presence(
                model, tokenizer, label=label, n=N_MAIN, table=table, fields=state,
                seed=gen_seed(s, g), hold_suffix=(MODE == "held"), stats=stats,
                decoder="temperature", decoder_kwargs={"temperature": 1.0},
                field_bias={"PTH": {tokenizer.eos_id: -DELTA[label]}}, device=device)
            path.write_text(json.dumps(urls), encoding="utf-8")
        except PresenceUnmet as exc:
            urls, error = [], str(exc)
    records.append({"file": path.name, "state_index": s, "label": label,
                    "state": state, "generation": g, "seed": gen_seed(s, g),
                    "n": len(urls), "stats": stats, "error": error,
                    "sha256": sha256(path) if path.is_file() else None,
                    "seconds": round(time.time() - t0, 1)})
    left = (time.time() - started) / k * (len(jobs) - k)
    print(f"[{k}/{len(jobs)}] {path.name}: {len(urls)} urls, draws {stats.get('draws')} "
          f"in {(time.time() - t0) / 60:.1f} min (about {left / 60:.0f} min left)"
          + (f"  ERROR {error}" if error else ""), flush=True)

In [ ]:
# --- 5. provenance ---------------------------------------------------------------
import platform

prov = {"purpose": "CR1 generations (research/cr1/PREREG_CR1.md); generation only",
        "mode": MODE, "G": G, "n_main": N_MAIN,
        "generation": {"decoder": "temperature", "temperature": 1.0, "delta": DELTA,
                       "frame": "domain", "seed_rule": "20261001 + (100*s + g) * 1e9"},
        "device": {"forward": str(device), "gpu": torch.cuda.get_device_name(0),
                   "sampling": "cpu (seeded torch.Generator, as in generate())"},
        "environment": {"torch": torch.__version__, "python": platform.python_version(),
                        "tldextract": metadata.version("tldextract")},
        "code_commit": manifest["commit"],
        "checkpoint": {"source": str(CKPT), "sha256": sha256(CKPT)},
        "suffix_table_sha256": TABLE_SHA256, "files": records,
        "total_minutes": round((time.time() - started) / 60, 1)}
(OUT / "PROVENANCE.json").write_text(json.dumps(prov, indent=1), encoding="utf-8")
ok = all(r["n"] == N_MAIN and r["error"] is None for r in records)
print(f"files {len(records)}; all complete: {ok}")
print("DONE" if ok else "INCOMPLETE -- see the ERROR lines above")

---

**Download** the *Output* tab (`cr1_gen/`: 70 files and `PROVENANCE.json`) into
`Downloads`. On the laptop, `research/cr1/cr1_verify.py` checks every file name,
count, hash and state, and regenerates a sample on the CPU for byte identity,
before the gate's scorer reads anything.